This notebook allows for robustness analyses of our sample and results, using post-stratification weights for gender, age, and geography based on ISTAT 2025 data

In [138]:
import pandas as pd
from scipy import stats
from scipy.stats import norm
import pingouin as pg
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import chi2_contingency
from collections import Counter
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
sns.set_theme("notebook", style="whitegrid", font_scale=1.3)

In [139]:
#load the data and skip second row (translated header)
df_clean = pd.read_csv("/home/XXX4/postdoc/it_survey/rita_survey/survey_v3_clean.tsv", sep="\t", skiprows=[1], encoding='utf-8')
df.head(3)

,StartDate,EndDate,Status,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,DistributionChannel,UserLanguage,Q_RecaptchaScore,Q_RelevantIDDuplicate,Q_DuplicateRespondent,Q_RelevantIDDuplicateScore,Q_RelevantIDFraudScore,Q_RelevantIDLastStartDate,Q14,Q1,Q6,Q17,Q17_6_TEXT,Istruzione,Q20,Q26,Q26_15_TEXT,Q10,Q10_2_TEXT,Q16,Q20-MT,Q20-MT_6_TEXT,Q21,Q21_8_TEXT,Q33,Q33_7_TEXT,Q34,Q34_8_TEXT,Q36,Q36_9_TEXT,Q43,Q43_9_TEXT,Q51_1,Q51_2,Q51_3,Q51_4,Q51_5,Q51_6,Q52,Q52_22_TEXT,Q53_1,Q53_2,Q53_3,Q53_4,Q53_5,Q53_6,Q53_7,Q53_8,Q53_9,Q53_10,Q53_11,Q53_12,Q53_13,Q53_14,Q53_15,Q53_16,Q53_17,Q53_18,Q53_19,Q53_20,Q53_21,Q53_22,Q53_23,Q53_24,Q53_25,Q53_25_TEXT,access,access_5_TEXT,language_used,language_used_3_TEXT,Q46,interaction_mode,interaction_mode_4_TEXT,Q48,strategies,strategies_5_TEXT,verification,Q54,Q61,Q62,Q63,Q64,Q65,Q66,Q67,Q67_7_TEXT,prior_knowledge,education_technology,likert_1_1,likert_1_3,likert_1_4,likert_1_5,likert_1_6,likert_2_1,likert_2_2,likert_2_3,personal_experience,Q85,never_used,never_used_5_TEXT,reason_never_used,reason_never_used_4_TEXT,future_use,future_use_7_TEXT,Q_DataPolicyViolations,Q_StraightliningCount,Q_StraightliningPercentage,Q_StraightliningQuestions,Q_UnansweredPercentage,Q_UnansweredQuestions,mapped_job,weight
0,2025-05-23 2:23:05,2025-05-23 2:29:09,IP Address,100,363,True,2025-05-23 2:29:09,R_8zCIYRTtI5tvimW,anonymous,IT,0.9,NaN,NaN,0.0,0.0,NaN,Sì (partecipo allo studio),Madrelingua,25-34 anni,Uomo,NaN,Laurea magistrale o master di primo livello,Area scientifico-tecnologica,Impresa e consulenza aziendale,NaN,"Nord-Est, Italia",NaN,Medio Bassa,"Google Traduttore,DeepL",NaN,Siri,NaN,Grammarly,NaN,Trascrizione automatica su Zoom/Google Meet/Te...,NaN,Non ho mai usato applicazioni di produzione di...,NaN,"ChatGPT,Gemini,Claude,Copilot,Perplexity,Mid-J...",NaN,Mai,Almeno una volta al mese,Almeno una volta a settimana,Almeno una volta a settimana,Meno di una volta al mese,Almeno una volta al mese,"Scrittura di email,Altra scrittura,Raccolta id...",NaN,Studio/Lavoro,NaN,NaN,Studio/Lavoro,NaN,Studio/Lavoro,Studio/Lavoro,Studio/Lavoro,NaN,NaN,NaN,NaN,NaN,Svago/Uso personale,NaN,NaN,NaN,NaN,NaN,NaN,Studio/Lavoro,NaN,NaN,NaN,NaN,NaN,Interfaccia Web da computer,NaN,"Italiano,Inglese",NaN,NaN,Principalmente scrivendo,NaN,NaN,Fornisco istruzioni dettagliate fin da subito,NaN,"Sempre, mi voglio assicurare che le risposte s...",Sì,Parzialmente sostituito,Parzialmente sostituito,Parzialmente sostituito,Parzialmente sostituito,NaN,Parzialmente sostituito,Più comodo,NaN,Sì,"Sì, sul posto di lavoro/a scuola",3,3,3,4,4,4,4,3,Sì,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,NaN,0.078947,"QID46,QID48,QID85",Impresa e consulenza aziendale,0.49
1,2025-05-23 2:27:17,2025-05-23 2:32:43,IP Address,100,325,True,2025-05-23 2:32:43,R_2ZE9jb0FUPux0tt,anonymous,IT,0.8,NaN,NaN,0.0,0.0,NaN,Sì (partecipo allo studio),Madrelingua,25-34 anni,Donna,NaN,Laurea magistrale o master di primo livello,Area economico-giuridica,Finanza,NaN,"Nord-Ovest, Italia",NaN,Medio Alta,Google Traduttore,NaN,Non ho mai usato applicazioni di assistenti vo...,NaN,Non ho mai usato applicazioni di scrittura ass...,NaN,Non ho mai usato applicazioni di trascrizione ...,NaN,Non ho mai usato applicazioni di produzione di...,NaN,"ChatGPT,Gemini,Copilot",NaN,Almeno una volta al mese,Meno di una volta al mese,Almeno una volta al mese,Almeno una volta al mese,Mai,Mai,"Raccolta idee e creazione di bozze,Controllo d...",NaN,NaN,NaN,NaN,NaN,NaN,Studio/Lavoro,NaN,NaN,NaN,NaN,"Studio/Lavoro,Svago/Uso personale",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Studio/Lavoro,NaN,NaN,NaN,NaN,NaN,Interfaccia Web da computer,NaN,"Italiano,Inglese",NaN,NaN,Principalmente scrivendo,NaN,NaN,"Fornisco istruzioni dettagliate fin da subito,...",NaN,"A volte, verifico solo quando qualcosa mi semb...",No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sì,No,1,1,3,3,2,4,2,4,Sì,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,NaN,0.093750,"QID46,QID48,QID85",Finanza,0.58
2,2025-05-23 2:33:42,2025-05-23 2:39:23,IP Address,100,341,T

In [140]:
# Discard NA values for chatbot adoption question
df_clean = df_clean[df_clean['Q43'].notna()].copy()
len(df_clean)

1906

In [141]:
pd.set_option("display.max_rows", None)  
pd.set_option("display.max_columns", None) 

In [142]:
# check weights for each variables combo
var = ['Q6', 'Q17', 'Q10', 'weight']
combos = df[var].drop_duplicates().sort_values('weight')
print(combos)

                     Q6                        Q17                 Q10  weight
6            25-34 anni                      Donna    Nord-Est, Italia    0.28
37           18-24 anni                      Donna    Nord-Est, Italia    0.32
0            25-34 anni                       Uomo    Nord-Est, Italia    0.49
7            55-64 anni                      Donna    Nord-Est, Italia    0.53
29    Dai 65 anni in su                      Donna    Nord-Est, Italia    0.54
26           25-34 anni                       Uomo  Nord-Ovest, Italia    0.57
1            25-34 anni                      Donna  Nord-Ovest, Italia    0.58
401          55-64 anni                      Donna       Isole, Italia    0.60
21    Dai 65 anni in su                       Uomo    Nord-Est, Italia    0.61
95           25-34 anni                      Donna       Centro Italia    0.61
116          55-64 anni                       Uomo  Nord-Ovest, Italia    0.64
103          25-34 anni                       Uomo  

In [185]:
# # Weight check
# print("N responses:", len(df))
# print("Weight sum", df['weight'].sum())

# Variables

In [144]:
# Mapping education
education_map = {k: 'Graduates' for k in [
    'Laurea magistrale o master di primo livello',
    'Laurea triennale o a ciclo unico',
    'Dottorato di ricerca',
    'Master di secondo livello'
]}
education_map.update({k: 'Non-graduates' for k in [
    'Diploma di scuola superiore',
    'Istruzione secondaria di primo grado (medie)', 
    'Istruzione primaria (elementari)'
]})

# Mapping geography
def geography_map(x):
    if x in ['Nord-Ovest, Italia', 'Nord-Est, Italia']:
        return 'North'
    elif x in ['Sud Italia', 'Isole, Italia']:
        return 'South and Islands'
    elif x == 'Centro Italia':
        return 'Centre'
    elif x == 'Non vivo in Italia':
        return 'Abroad'

# Mapping gender
def gender_map(x):
    if x == 'Uomo':
        return 'Man'
    elif x == 'Donna':
        return 'Woman'
    else:
        return 'Neither'

# Mapping income
def income_map(x):
    if x in ['Bassa', 'Medio Bassa']:
        return 'Lower'
    elif x in ['Medio Alta', 'Alta']:
        return 'Higher'
    elif x == 'Media':
        return 'Mid'

def age_map(x):
    if x in ['18-24 anni', '25-34 anni']:
        return '18-34'
    elif x in ['35-44 anni', '45-54 anni']:
        return '35-54'
    elif x == '55-64 anni':
        return '55-64'
    else:
        return '65+'

In [145]:
df_clean['EducationGroup'] = df_clean['Istruzione'].map(education_map)
df_clean['GeographyGroup'] = df_clean['Q10'].apply(geography_map)
df_clean['GenderGroup'] = df_clean['Q17'].apply(gender_map)
df_clean['IncomeGroup'] = df_clean['Q16'].apply(income_map)
df_clean['AgeGroup'] = df_clean['Q6'].apply(age_map)

In [146]:
# Column for chatbot usage
chatbot_col = 'Q43'

# Check if column exists
if chatbot_col in df_clean.columns:
    # Create chatbot_user column
    # If contains "Non ho mai usato" = No (not a user)
    # Otherwise = Yes (is a user)
    df_clean['chatbot_user'] = df_clean['Q43'].str.contains("Non ho mai usato", case=False, na=False).map({True: 'No', False: 'Yes'})

In [147]:
# Intent and Frequency
intent_mapping = {
    'Q51_1': 'InfoRetrieval',
    'Q51_2': 'ProblemSolving',
    'Q51_3': 'Learning',
    'Q51_4': 'ContentCreation',
    'Q51_5': 'Entertainment',
    'Q51_6': 'Creativity'
}

# Define frequency order
freq_order = ["Mai", "Meno di una volta al mese", "Almeno una volta al mese", "Almeno una volta a settimana"]

# Create numerical mapping
freq_to_num = {
    "Mai": 0,
    "Meno di una volta al mese": 1,
    "Almeno una volta al mese": 2,
    "Almeno una volta a settimana": 3
}

# Convert each intent column to numerical frequency in df_clean
for old_col, new_col in intent_mapping.items():
    if old_col in df_clean.columns:
        # Create numerical version (will be NaN for non-users who don't have responses)
        df_clean[f'{new_col}_freq'] = df_clean[old_col].map(freq_to_num)

# Checks definitions 

In [148]:
def freq_table(df, col, weight_col='weight', categories=None, dropna=True):
    """
    Compute frequencies, unweighted and weighted percentages for a categorical column,
    along with delta and cumulative percentages.
    """
    col_data = df[col]
    w = df[weight_col]

    # Determine categorical responses
    if categories is None:
        categories = sorted(col_data.dropna().unique())
    cat = pd.Categorical(col_data, categories=categories, ordered=False)

    # Unweighted frequencies
    freq_unw = cat.value_counts(dropna=dropna).rename('unw_freq')

    # Weighted frequencies (valid rows only)
    mask_valid = ~cat.isna()
    wfreq = (
        pd.Series(w[mask_valid].values, index=cat[mask_valid])
        .groupby(level=0, observed=True).sum()
        .reindex(categories)
        .fillna(0)
        .rename('w_freq')
    )

    # Percentages
    unw_pct = (freq_unw / (~cat.isna()).sum()).rename('unw_pct')
    # denominator = total weight of valid responses
    den = w[mask_valid].sum()  
    wt_pct = (wfreq / den).rename('w_pct')

    # 
    out = pd.concat([freq_unw, wfreq, unw_pct, wt_pct], axis=1).fillna(0)
    out['delta_pp'] = (out['w_pct'] - out['unw_pct']) * 100
    out['cum_unw'] = out['unw_pct'].cumsum() * 100
    out['cum_w'] = out['w_pct'].cumsum() * 100

    # Convert percentages to %
    out[['unw_pct', 'w_pct']] = out[['unw_pct', 'w_pct']] * 100

    return out.round(2)

In [149]:
# Flag potential bias based on predefined thresholds
def question_flag(table, delta_pp_thresholds=(3,5)):
    dmax = table['delta_pp'].abs().max()
    if dmax <= delta_pp_thresholds[0]:
        flag = 'PASS'
    elif dmax <= delta_pp_thresholds[1]:
        flag = 'REVIEW'
    else:
        flag = 'CRITICAL'
    #print(f"Question flag: {flag}, delta_pp={dmax:.2f}")
    return flag, dmax

In [151]:
def batch_dashboard(df, questions, weight_col='weight', categories_map=None):
    rows = []
    # Get categorical responses
    for q in questions:
        cats = None if categories_map is None else categories_map.get(q)
        # Compute the frequency table for this question
        tab = freq_table(df, q, weight_col, categories=cats)
        flag, dmax = question_flag(tab)
        # Check if the top-2 responses match (1) or not (0) between unweighted and weighted percentages
        top2_unw = tab['unw_pct'].sort_values(ascending=False).index[:2].tolist()
        top2_w   = tab['w_pct'].sort_values(ascending=False).index[:2].tolist()
        top2_same = int(top2_unw == top2_w)
        rows.append({'question': q, 'delta_max_pp': float(dmax),
                     'top2_same': top2_same, 'flag': flag})
    # sort by flag and delta
    dash = pd.DataFrame(rows).sort_values(['flag','delta_max_pp'], ascending=[True, False])
    return dash

# Apply robustness checks

## Education

In [155]:
# Education
questions = ['EducationGroup']
dashboard = batch_dashboard(df_clean, questions)
print(dashboard)

         question  delta_max_pp  top2_same  flag
0  EducationGroup          0.18          1  PASS


In [156]:
edu_table = freq_table(df_clean, 'EducationGroup', weight_col='weight')
print(chatbot_table)

     unw_freq   w_freq  unw_pct  w_pct  delta_pp  cum_unw   cum_w
No        373   380.45    19.57  19.92      0.35    19.57   19.92
Yes      1533  1529.86    80.43  80.08     -0.35   100.00  100.00


## Chatbot users

In [157]:
questions = ['chatbot_user']
dashboard = batch_dashboard(df_clean, questions)
print(dashboard)

       question  delta_max_pp  top2_same  flag
0  chatbot_user          0.35          1  PASS


In [158]:
chatbot_table = freq_table(df_clean, 'chatbot_user', weight_col='weight')
print(chatbot_table)

     unw_freq   w_freq  unw_pct  w_pct  delta_pp  cum_unw   cum_w
No        373   380.45    19.57  19.92      0.35    19.57   19.92
Yes      1533  1529.86    80.43  80.08     -0.35   100.00  100.00


## Socioeconomic

In [159]:
questions = ['IncomeGroup']
dashboard = batch_dashboard(df_clean, questions)
print(dashboard)

      question  delta_max_pp  top2_same  flag
0  IncomeGroup          1.46          1  PASS


In [160]:
income_table = freq_table(df_clean, 'IncomeGroup', weight_col='weight')
print(chatbot_table)

     unw_freq   w_freq  unw_pct  w_pct  delta_pp  cum_unw   cum_w
No        373   380.45    19.57  19.92      0.35    19.57   19.92
Yes      1533  1529.86    80.43  80.08     -0.35   100.00  100.00


## LT adoption and replacement

In [161]:
# Define mappings to calculate proportions relative the N of users of the specific application
lt_mapping = {
    'Q61': 'Q20-MT',             # MT
    'Q62': 'Q34',                # Speech Transcript
    'Q63': 'Q21',                # Vocal Assistants
    'Q64': 'Q33',                # Assisted Writing
    'Q65': 'Q36',                # Text-to-Speech
    'Q66': 'Q43'                 # Web Search replacement calculated among GenAI users
}

In [162]:
# Create binary usage columns (1 = uses, 0 = doesn't use)
for rep_q, use_q in lt_mapping.items():
    if use_q:
        used_mask = ~df_clean[use_q].str.contains("Non ho mai usato", na=False, case=False)
        df_clean[f'{rep_q}_uses_tech'] = used_mask.astype(int)
        df_clean.loc[~used_mask, f'{rep_q}_uses_tech'] = 0
    else:
        df_clean[f'{rep_q}_uses_tech'] = 1  


# Create binary replacement columns (among users only)
for rep_q, use_q in lt_mapping.items():
    if use_q:
        used_mask = ~df_clean[use_q].str.contains("Non ho mai usato", na=False, case=False)
    else:
        used_mask = df_clean[rep_q].notna()
    
    # Completely replaced (binary: 1 = yes, 0 = no, NaN = non-user)
    df_clean[f'{rep_q}_comp_replaced'] = np.nan
    df_clean.loc[used_mask, f'{rep_q}_comp_replaced'] = (
        df_clean.loc[used_mask, rep_q] == "Completamente sostituito"
    ).astype(int)
    
    # Partially replaced (binary: 1 = yes, 0 = no, NaN = non-user)
    df_clean[f'{rep_q}_parz_replaced'] = np.nan
    df_clean.loc[used_mask, f'{rep_q}_parz_replaced'] = (
        df_clean.loc[used_mask, rep_q] == "Parzialmente sostituito"
    ).astype(int)

usage_cols = [f'{rep_q}_uses_tech' for rep_q in lt_mapping.keys()]
comp_cols = [f'{rep_q}_comp_replaced' for rep_q in lt_mapping.keys()]
parz_cols = [f'{rep_q}_parz_replaced' for rep_q in lt_mapping.keys()]

In [163]:
usage_dashboard = batch_dashboard(df_clean, usage_cols)
comp_dashboard = batch_dashboard(df_clean, comp_cols)
parz_dashboard = batch_dashboard(df_clean, parz_cols)

In [164]:
print(usage_dashboard)

        question  delta_max_pp  top2_same  flag
3  Q64_uses_tech          2.19          1  PASS
4  Q65_uses_tech          1.22          1  PASS
0  Q61_uses_tech          0.87          1  PASS
2  Q63_uses_tech          0.64          1  PASS
5  Q66_uses_tech          0.35          1  PASS
1  Q62_uses_tech          0.24          1  PASS


In [165]:
print(comp_dashboard)

            question  delta_max_pp  top2_same  flag
3  Q64_comp_replaced          3.00          1  PASS
4  Q65_comp_replaced          2.13          1  PASS
1  Q62_comp_replaced          0.72          1  PASS
0  Q61_comp_replaced          0.65          1  PASS
2  Q63_comp_replaced          0.62          1  PASS
5  Q66_comp_replaced          0.26          1  PASS


In [166]:
print(parz_dashboard)

            question  delta_max_pp  top2_same    flag
1  Q62_parz_replaced          1.94          1    PASS
4  Q65_parz_replaced          1.33          1    PASS
2  Q63_parz_replaced          1.14          1    PASS
0  Q61_parz_replaced          0.15          1    PASS
5  Q66_parz_replaced          0.01          1    PASS
3  Q64_parz_replaced          4.51          1  REVIEW


In [167]:
print("\nQ64 Parz Replaced:")
print(freq_table(df_clean, 'Q64_parz_replaced', weight_col='weight'))


Q64 Parz Replaced:
     unw_freq  w_freq  unw_pct  w_pct  delta_pp  cum_unw   cum_w
0.0       286  239.79    75.46  70.95     -4.51    75.46   70.95
1.0        93   98.18    24.54  29.05      4.51   100.00  100.00


## Check reasons for replacement

In [168]:
questions = ['Q67']
dashboard = batch_dashboard(df, questions)
print(dashboard)

  question  delta_max_pp  top2_same  flag
0      Q67          2.78          1  PASS


## Intent Frequency

In [169]:
# Get all frequency columns
freq_columns = [f'{new_col}_freq' for new_col in intent_mapping.values()]

intent_dashboard = batch_dashboard(df_clean, freq_columns)
print(intent_dashboard)

               question  delta_max_pp  top2_same    flag
3  ContentCreation_freq          2.22          1    PASS
1   ProblemSolving_freq          1.65          1    PASS
0    InfoRetrieval_freq          1.54          0    PASS
5       Creativity_freq          1.54          1    PASS
4    Entertainment_freq          0.85          1    PASS
2         Learning_freq          3.02          1  REVIEW


In [170]:
print(freq_table(df_clean, 'Learning_freq', weight_col='weight'))

     unw_freq  w_freq  unw_pct  w_pct  delta_pp  cum_unw   cum_w
0.0       166  162.95    11.42  11.36     -0.06    11.42   11.36
1.0       251  250.59    17.26  17.47      0.20    28.68   28.82
2.0       308  260.55    21.18  18.16     -3.02    49.86   46.98
3.0       729  760.62    50.14  53.02      2.88   100.00  100.00


## Work and personal use by occupation

In [171]:
# Activity labels (shortened for better display)
q53_to_label_en = {
    "Q53_1": "Email Writing", "Q53_2": "Creative Writing", "Q53_3": "Academic Writing",
    "Q53_4": "Other Writing", "Q53_5": "Text Analysis", "Q53_6": "Idea Generation",
    "Q53_7": "Explanations", "Q53_8": "Summaries", "Q53_9": "Quiz Creation",
    "Q53_10": "Travel Planning", "Q53_11": "Fact Checking", "Q53_12": "Personal Advice",
    "Q53_13": "Medical Advice", "Q53_14": "Other Advice", "Q53_15": "Emotional Support",
    "Q53_16": "Casual Chat", "Q53_17": "Romantic Chat", "Q53_18": "Philosophical Chat",
    "Q53_19": "Other Chat", "Q53_20": "Programming Help", "Q53_21": "Data Analysis",
    "Q53_22": "Image Generation", "Q53_23": "Audio/Music Gen.", "Q53_25": "Other"
}

In [172]:
q53_columns = [f'Q53_{i}' for i in range(1, 26) if i != 24]

# Calculate totals for each person
df_clean['total_work_responses'] = 0
df_clean['total_personal_responses'] = 0

for col in q53_columns:
    if col in df_clean.columns:
        df_clean['total_work_responses'] += df_clean[col].fillna('').str.contains('Studio/Lavoro', case=False).astype(int)
        df_clean['total_personal_responses'] += df_clean[col].fillna('').str.contains('Svago/Uso personale', case=False).astype(int)

# For each occupation, create a work/personal column (only for chatbot users)
if 'mapped_job' in df_clean.columns and 'chatbot_user' in df_clean.columns:
    chatbot_mask = df_clean['chatbot_user'] == 'Yes'
    
    for occ_it, occ_en in extended_occ_mapping.items():
        if occ_it not in ['Altro', 'Agricoltura e ambiente']:
            col_name = f'usage_type_{occ_en.replace(" ", "_").replace("&", "and")}'
            
            # Initialize as NaN
            df_clean[col_name] = np.nan
            
            # For this occupation's chatbot users, assign Work or Personal based on majority
            occ_mask = chatbot_mask & (df_clean['mapped_job'] == occ_it)
            has_responses = (df_clean['total_work_responses'] + df_clean['total_personal_responses']) > 0
            
            # Assign "Work" if work responses >= personal, else "Personal"
            work_majority = df_clean['total_work_responses'] >= df_clean['total_personal_responses']
            
            df_clean.loc[occ_mask & has_responses & work_majority, col_name] = 'Work'
            df_clean.loc[occ_mask & has_responses & ~work_majority, col_name] = 'Personal'

# Get all occupation usage columns
occupation_usage_cols = [f'usage_type_{occ_en.replace(" ", "_").replace("&", "and")}' 
                         for occ_it, occ_en in extended_occ_mapping.items()
                         if occ_it not in ['Altro', 'Agricoltura e ambiente']]

In [173]:
# Use your dashboard
usage_dashboard = batch_dashboard(df_clean, occupation_usage_cols)
print(usage_dashboard)

                                question  delta_max_pp  top2_same      flag
8                  usage_type_Healthcare         15.25          1  CRITICAL
12   usage_type_Services_and_Hospitality          5.61          1  CRITICAL
2                   usage_type_Education          2.57          1      PASS
3       usage_type_Research_and_Academia          1.52          1      PASS
5   usage_type_Security_and_Public_Admin          0.96          1      PASS
11  usage_type_Culture_and_Entertainment          0.84          1      PASS
7      usage_type_Industry_and_Transport          0.19          1      PASS
6                     usage_type_Student          4.70          1    REVIEW
9       usage_type_Currently_Not_Working          3.99          0    REVIEW
4     usage_type_Business_and_Consulting          3.93          1    REVIEW
0                     usage_type_Retired          3.68          1    REVIEW
13               usage_type_Construction          3.35          1    REVIEW
10          

In [174]:
print(freq_table(df_clean, 'usage_type_Healthcare', weight_col='weight'))

          unw_freq  w_freq  unw_pct  w_pct  delta_pp  cum_unw   cum_w
Personal        31   20.85    41.89  26.64    -15.25    41.89   26.64
Work            43   57.41    58.11  73.36     15.25   100.00  100.00


In [175]:
print(freq_table(df_clean, 'usage_type_Services_and_Hospitality', weight_col='weight'))

          unw_freq  w_freq  unw_pct  w_pct  delta_pp  cum_unw  cum_w
Personal        19   21.85    54.29   59.9      5.61    54.29   59.9
Work            16   14.63    45.71   40.1     -5.61   100.00  100.0


## Usage modalities and strategies

In [176]:
questions = ['strategies']
dashboard = batch_dashboard(df, questions)
print(dashboard)

     question  delta_max_pp  top2_same  flag
0  strategies          1.98          0  PASS


In [177]:
questions = ['access', 'interaction_mode']
dashboard = batch_dashboard(df, questions)
print(dashboard)

           question  delta_max_pp  top2_same  flag
0            access          1.87          1  PASS
1  interaction_mode          0.54          1  PASS


## Language Used

In [178]:
# Get all unique languages
all_languages = (df_clean['language_used'].dropna().str.split(',').explode().str.strip().unique())

# Create binary column for each language (1 = uses this language, 0 = doesn't, NaN = no answer)
for lang in all_languages:
    col_name = f'uses_{lang.replace(" ", "_")}'
    
    # Initialize as NaN
    df_clean[col_name] = np.nan
    
    # For people who answered language_used
    has_answer = df_clean['language_used'].notna()
    
    # 1 if this language is in their list, 0 if not
    df_clean.loc[has_answer, col_name] = df_clean.loc[has_answer, 'language_used'].str.contains(
        lang, case=False, na=False, regex=False
    ).astype(int)

# Get all language columns
language_cols = [f'uses_{lang.replace(" ", "_")}' for lang in all_languages]

In [179]:
# Use your dashboard
language_dashboard = batch_dashboard(df_clean, language_cols)
print(language_dashboard)

        question  delta_max_pp  top2_same      flag
1   uses_Inglese          5.06          0  CRITICAL
0  uses_Italiano          0.78          1      PASS
2     uses_Altro          0.36          1      PASS
3    uses_Non_so          0.02          1      PASS


In [180]:
print(freq_table(df_clean, 'uses_Inglese', weight_col='weight'))

     unw_freq  w_freq  unw_pct  w_pct  delta_pp  cum_unw   cum_w
0.0       728  803.16    48.37  53.43      5.06    48.37   53.43
1.0       777  700.00    51.63  46.57     -5.06   100.00  100.00


## Experience with errors and Bias

In [181]:
questions = ['personal_experience']
dashboard = batch_dashboard(df, questions)
print(dashboard)

              question  delta_max_pp  top2_same  flag
0  personal_experience          1.58          1  PASS


## LT literacy and desiderata

In [182]:
questions = ['likert_1_1', 'likert_1_3', 'likert_1_4', 'likert_1_5', 'likert_1_6', 'likert_2_1', 'likert_2_2', 'likert_2_3']
dashboard = batch_dashboard(df, questions)
print(dashboard)

     question  delta_max_pp  top2_same  flag
5  likert_2_1          2.89          1  PASS
6  likert_2_2          2.01          1  PASS
2  likert_1_4          1.53          0  PASS
0  likert_1_1          1.43          1  PASS
3  likert_1_5          1.09          1  PASS
7  likert_2_3          0.68          0  PASS
1  likert_1_3          0.62          0  PASS
4  likert_1_6          0.57          1  PASS


## Prior LT education

In [183]:
questions = ['education_technology']
dashboard = batch_dashboard(df, questions)
print(dashboard)

               question  delta_max_pp  top2_same  flag
0  education_technology          1.18          0  PASS


## Questions for non-users

In [184]:
questions = ['never_used', 'reason_never_used', 'future_use']
dashboard = batch_dashboard(df, questions)
print(dashboard)

            question  delta_max_pp  top2_same  flag
0         never_used          2.91          0  PASS
1  reason_never_used          1.72          1  PASS
2         future_use          1.19          1  PASS
